<a href="https://colab.research.google.com/github/PraveenKumar-pk-star/DS-corrected-file/blob/main/Spread_sheet__Itilite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-community
!pip install -q pydantic
!pip install -q openai
!pip install -q gspread oauth2client
!pip install -q pymupdf pytesseract pillow

In [69]:
import os
import getpass
import fitz
import pytesseract
import gspread

from PIL import Image
from google.colab import files
from pydantic import BaseModel
from oauth2client.service_account import ServiceAccountCredentials

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


In [70]:
os.environ["OPENAI_API_KEY"] = getpass.getpass(
    "Enter OpenAI API Key: "
)

Enter OpenAI API Key: ··········


In [71]:

uploaded = files.upload()

pdf_file_name = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_file_name)


Saving Giorgi 01 sv.pdf to Giorgi 01 sv.pdf
Uploaded PDF: Giorgi 01 sv.pdf


In [72]:
full_text = ""

pdf_document = fitz.open(pdf_file_name)

for page_num in range(len(pdf_document)):

    page = pdf_document[page_num]

    # Convert PDF page to image
    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))

    image_path = f"page_{page_num}.png"

    pix.save(image_path)

    # OCR
    image = Image.open(image_path)

    text = pytesseract.image_to_string(image)

    full_text += text + "\n"


print("\n================ PDF TEXT ================\n")
print(full_text[:3000])




================ PDF TEXT ================

 

@ Hotel Voucher Gitilite

Trip ID: 1991-0622 | Booked On: 01 May 2026 11:38 PM

Accommodation Details - Fri, 01 May 2026

 

 

 

 

Best Western Dickson Check-In Check-Out
2338 Highway 46 South 01-May-2026 07-May-2026
Dickson Tennessee 03:00 PM 11:00 AM
US - 37055
+1 615-446-0541
Travelers Breakfast Room Type Booking ID
Giorgi Sprow Included King Room - Non-Smoking 998072202
ize ac DL=lee) Ic} unt (USD)
6 Night(s) 464.96
Hotel Taxes and Charges 97.74
Total Amount 562.70

* Please Note: This reservation is already paid for, please do not pay at the hotel. For any payment
related issues, contact support immediately.

Guarantee & Cancellation Policy

* The full amount will be charged to your company's card at the time of booking.

* If you do not check in to the hotel on the first day of your reservation and do not alert the hotel in
advance, the hotel reserves the right to cancel your reservation and you may be charged for the full
amount

In [73]:
class BookingDetails(BaseModel):

    guest_name: str | None = None
    hotel_name: str | None = None
    confirmation_id: str | None = None
    check_in_date: str | None = None
    check_out_date: str | None = None


parser = PydanticOutputParser(
    pydantic_object=BookingDetails
)


In [74]:
prompt = PromptTemplate(

    template="""

You are an expert hotel booking receipt extractor.

Extract hotel booking information from hotel vouchers,
hotel invoices, hotel receipts, or reservation PDFs.

Required Fields:
- guest_name
- hotel_name
- confirmation_id
- check_in_date
- check_out_date

Instructions:

- Hotel PDFs can have any layout
- Guest name may appear as:
  guest, traveller, traveler, passenger

- Confirmation ID may appear as:
  confirmation number,
  booking id,
  reservation id,
  itinerary number

- Check-in date may appear as:
  check in, arrival date

- Check-out date may appear as:
  check out, departure date

Rules:
- Return structured output only
- If missing return null
- Do not make up information

{format_instructions}

HOTEL PDF TEXT:
{text}

""",

    input_variables=["text"],

    partial_variables={
        "format_instructions":
        parser.get_format_instructions()
    }
)

In [75]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [76]:
chain = prompt | llm | parser

In [77]:
result = chain.invoke({
    "text": full_text
})

In [78]:
print("\n================ EXTRACTED DATA ================\n")

print("Guest Name      :", result.guest_name)
print("Hotel Name      :", result.hotel_name)
print("Confirmation ID :", result.confirmation_id)
print("Check In Date   :", result.check_in_date)
print("Check Out Date  :", result.check_out_date)


================ EXTRACTED DATA ================

Guest Name      : Giorgi Sprow
Hotel Name      : Best Western Dickson
Confirmation ID : 998072202
Check In Date   : 01-May-2026
Check Out Date  : 07-May-2026


In [79]:
result_dict = result.dict()

print("\n================ JSON OUTPUT ================\n")

print(result_dict)


================ JSON OUTPUT ================

{'guest_name': 'Giorgi Sprow', 'hotel_name': 'Best Western Dickson', 'confirmation_id': '998072202', 'check_in_date': '01-May-2026', 'check_out_date': '07-May-2026'}


/tmp/ipykernel_1372/1852542960.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  result_dict = result.dict()


In [80]:

print("\nUpload Google Service Account JSON File")

uploaded = files.upload()


Upload Google Service Account JSON File


Saving excelautomation-496518-dd872357982a.json to excelautomation-496518-dd872357982a (3).json


In [81]:
scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

json_file_name = list(uploaded.keys())[0]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    json_file_name,
    scope
)

client = gspread.authorize(creds)

In [82]:
sheet = client.open_by_url(
    "https://docs.google.com/spreadsheets/d/1PNtZvdpU-91QrZWcmnBYFXzZhsNhx5vJIhb-A7qopoc/edit?usp=sharing"
).sheet1

In [83]:
row = [

    result.guest_name,
    result.hotel_name,
    result.confirmation_id,
    result.check_in_date,
    result.check_out_date

]

sheet.append_row(row)

print("\n✅ Data uploaded successfully!")


✅ Data uploaded successfully!
